# CMPE 249 detection: Stage 2 runner
Runtime → Change runtime type → **L4 GPU** (use the same GPU type for every latency number).

No Google Drive: data lives on Colab local disk, and results are pushed to this repo's `results` branch after every run.
One-time setup: add a GitHub token as a Colab Secret named `GITHUB_TOKEN` (key icon in the left sidebar), with
*Contents: read and write* on `shloakk/2d-object-detection` only.

In [ ]:
!nvidia-smi

In [ ]:
!git clone -q https://github.com/shloakk/2d-object-detection.git /content/2d-object-detection
%cd /content/2d-object-detection
!git log -1 --oneline
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
# Results go into a clone of the `results` branch; src.sync_results commits and pushes it.
!git clone -q -b results https://{token}@github.com/shloakk/2d-object-detection.git /content/results
!git -C /content/results config user.name 'Colab runner'
!git -C /content/results config user.email 'colab@users.noreply.github.com'
os.environ['CMPE249_DATA'] = '/content/data'        # Colab local disk: fast reads
os.environ['CMPE249_WEIGHTS'] = '/content/weights'
os.environ['CMPE249_RESULTS'] = '/content/results'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Data prep (once per session; local disk is wiped when the runtime ends)
!python -m src.data.kitti_to_coco            # full KITTI val (1496 images)
!python -m src.data.coco_subset --full       # full COCO val2017 (5000 images, fetched by coco_url)

In [ ]:
# One config end to end, then push its results to the `results` branch
!python -m src.predict --config configs/full/yolo26s_kitti.yaml && python -m src.sync_results

In [ ]:
# All accuracy runs (Faster R-CNN COCO is the slowest). Each run is pushed as soon as it finishes.
for cfg in ['yolo26s_coco', 'frcnn_v2_coco', 'rtdetr_l_coco', 'rtdetr_l_kitti', 'frcnn_v2_kitti']:
    !python -m src.predict --config configs/full/{cfg}.yaml && python -m src.sync_results

In [ ]:
# Latency: batch 1, 50 warm-up + 200 timed, FP32 and FP16, on 200 KITTI images
for m in ['yolo26s', 'rtdetr_l', 'frcnn_v2']:
    for half in ['', '--half']:
        !python -m src.bench_latency --config configs/full/{m}_kitti.yaml {half}
!python -m src.sync_results